In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
# ============================================================================
# CELL 1: MOUNT GOOGLE DRIVE & SETUP
# ============================================================================
import os
import sys
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path
import json
import pickle
from tqdm import tqdm
import warnings
warnings.filterwarnings('ignore')

# Set your dataset path in Google Drive
DATASET_PATH = '/content/drive/MyDrive/Tea Leaf Project/Original images.zip'  # Update this path
BASE_DIR = '/content/tea_leaf_fusion'

# Create working directory
os.makedirs(BASE_DIR, exist_ok=True)
os.chdir(BASE_DIR)

# Create folder structure
folders = ['data/raw_images', 'data/masks', 'data/features',
           'results', 'checkpoints', 'logs']
for folder in folders:
    os.makedirs(folder, exist_ok=True)

print("✓ Google Drive mounted")
print("✓ Working directory:", BASE_DIR)

# Check if dataset exists
if os.path.exists(DATASET_PATH):
    print(f"✓ Dataset found at: {DATASET_PATH}")
else:
    print(f"⚠ Dataset not found at: {DATASET_PATH}")
    print("Please update DATASET_PATH variable with your actual path")

In [ ]:
# ============================================================================
# UPDATED CELL 2: ROBUST EXTRACTION & FLATTENING
# ============================================================================
import zipfile
import shutil
from pathlib import Path

def extract_and_flatten(zip_path, extract_to='data/raw_images'):
    print(f"🚀 Starting extraction from: {zip_path}")

    if not os.path.exists(zip_path):
        print(f"❌ Error: File not found at {zip_path}")
        return False

    # Temporary directory for extraction
    temp_extract = 'data/temp_zip'
    os.makedirs(temp_extract, exist_ok=True)

    try:
        with zipfile.ZipFile(zip_path, 'r') as zip_ref:
            print(f"📦 Unzipping {len(zip_ref.namelist())} items...")
            zip_ref.extractall(temp_extract)

        # Define allowed extensions
        valid_extensions = ('.jpg', '.jpeg', '.png', '.bmp', '.tiff')

        # Recursively find all images
        print("🔍 Scanning for images in all subfolders...")
        all_found_images = []
        for root, dirs, files in os.walk(temp_extract):
            for file in files:
                if file.lower().endswith(valid_extensions):
                    all_found_images.append(os.path.join(root, file))

        print(f"📸 Found {len(all_found_images)} images. Moving to {extract_to}...")

        # Move and rename to avoid collisions if they have the same name in different folders
        for i, img_path in enumerate(all_found_images):
            img_name = os.path.basename(img_path)
            dest_path = os.path.join(extract_to, img_name)

            # If a file with this name already exists, prepend a number
            if os.path.exists(dest_path):
                dest_path = os.path.join(extract_to, f"{i}_{img_name}")

            shutil.move(img_path, dest_path)

        # Cleanup temp folder
        shutil.rmtree(temp_extract)

        # Final Verification
        final_images = [f for f in os.listdir(extract_to) if f.lower().endswith(valid_extensions)]
        print(f"✅ Success! {len(final_images)} images are now in {extract_to}")
        return True

    except Exception as e:
        print(f"❌ An error occurred: {e}")
        return False

# Execute
success = extract_and_flatten(DATASET_PATH)

# Verify dataset
image_files = list(Path('data/raw_images').glob('*.*'))
image_files = [f for f in image_files if f.suffix.lower() in ['.jpg', '.png', '.jpeg']]

print(f"\n{'='*60}")
print(f"FINAL STATUS: {len(image_files)} images ready for processing")
print(f"{'='*60}")

In [ ]:
# ============================================================================
# CELL 3: ADAPTIVE SEGMENTATION
# ============================================================================
import cv2
import numpy as np
from pathlib import Path
from tqdm import tqdm
import json

class AdaptiveLeafSegmenter:
    """Intelligent segmentation that adapts to background variability"""

    def __init__(self):
        self.mask_dir = Path('data/masks')
        self.mask_dir.mkdir(exist_ok=True)

    def estimate_background(self, image):
        """Estimate background from image borders"""
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        h, w = gray.shape

        # Sample from 10% border (usually background)
        border = int(min(h, w) * 0.1)

        top = gray[:border, :].flatten()
        bottom = gray[-border:, :].flatten()
        left = gray[:, :border].flatten()
        right = gray[:, -border:].flatten()

        bg_pixels = np.concatenate([top, bottom, left, right])

        if len(bg_pixels) > 0:
            bg_mean = np.mean(bg_pixels)
            bg_std = np.std(bg_pixels)
            return bg_mean, bg_std

        # Default values from your analysis
        return 70.38, 9.12

    def adaptive_segmentation(self, image):
        """Main segmentation algorithm"""
        gray = cv2.cvtColor(image, cv2.COLOR_BGR2GRAY)
        hsv = cv2.cvtColor(image, cv2.COLOR_BGR2HSV)

        # Get background stats
        bg_mean, bg_std = self.estimate_background(image)

        # Method 1: Adaptive threshold based on background
        if bg_mean < 50:  # Dark background
            thresh = bg_mean + 40
        elif bg_mean < 100:  # Medium
            thresh = bg_mean + 50
        else:  # Light background
            thresh = bg_mean + 60

        thresh = np.clip(thresh, 50, 200)
        _, mask1 = cv2.threshold(gray, thresh, 255, cv2.THRESH_BINARY)

        # Method 2: Use saturation channel
        saturation = hsv[:, :, 1]
        _, sat_mask = cv2.threshold(saturation, 30, 255, cv2.THRESH_BINARY)

        # Combine methods
        combined = cv2.bitwise_and(mask1, sat_mask)

        # Find and keep largest contour
        contours, _ = cv2.findContours(combined, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
        if contours:
            largest = max(contours, key=cv2.contourArea)
            final_mask = np.zeros_like(gray)
            cv2.drawContours(final_mask, [largest], -1, 255, -1)

            # Clean up
            kernel = np.ones((3, 3), np.uint8)
            final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_CLOSE, kernel)
            final_mask = cv2.morphologyEx(final_mask, cv2.MORPH_OPEN, kernel)

            return final_mask

        return combined

    def process_all_images(self, sample_limit=None, save_to_drive=False):
        """Process all images in directory"""
        image_dir = Path('data/raw_images')
        image_files = list(image_dir.glob('*.*'))
        image_files = [f for f in image_files if f.suffix.lower() in ['.jpg', '.png', '.jpeg']]

        if sample_limit:
            image_files = image_files[:sample_limit]

        print(f"Segmenting {len(image_files)} images...")
        results = []

        for img_path in tqdm(image_files, desc="Processing"):
            try:
                image = cv2.imread(str(img_path))
                if image is None:
                    continue

                mask = self.adaptive_segmentation(image)

                # Save mask
                mask_path = self.mask_dir / f"{img_path.stem}_mask.png"
                cv2.imwrite(str(mask_path), mask)

                results.append({
                    'image': img_path.name,
                    'mask': mask_path.name,
                    'mask_area': np.sum(mask > 0) / (mask.shape[0] * mask.shape[1])
                })

            except Exception as e:
                print(f"Error with {img_path.name}: {e}")

        # Save log
        with open('data/segmentation_log.json', 'w') as f:
            json.dump(results, f, indent=2)

        # Optionally save to Google Drive
        if save_to_drive:
            self._save_to_drive()

        print(f"\n✓ Segmented {len(results)} images")
        return results

    def _save_to_drive(self):
        """Save masks to Google Drive"""
        drive_masks_dir = '/content/drive/MyDrive/tea_leaf_masks'
        os.makedirs(drive_masks_dir, exist_ok=True)

        mask_files = list(self.mask_dir.glob('*.png'))
        for mask_file in tqdm(mask_files, desc="Saving to Drive"):
            shutil.copy(mask_file, f'{drive_masks_dir}/{mask_file.name}')

        print(f"✓ Masks saved to Google Drive: {drive_masks_dir}")

# Run segmentation (use sample_limit for testing)
print("Starting adaptive segmentation...")
segmenter = AdaptiveLeafSegmenter()

# For testing: process only 100 images
# seg_results = segmenter.process_all_images(sample_limit=100)

# For full dataset: process all images
seg_results = segmenter.process_all_images(save_to_drive=True)

# Show examples
import matplotlib.pyplot as plt
mask_files = list(Path('data/masks').glob('*mask.png'))[:3]
fig, axes = plt.subplots(1, 3, figsize=(15, 5))
for idx, mask_file in enumerate(mask_files[:3]):
    mask = cv2.imread(str(mask_file), cv2.IMREAD_GRAYSCALE)
    axes[idx].imshow(mask, cmap='gray')
    axes[idx].set_title(f"Mask: {mask_file.stem}")
    axes[idx].axis('off')
plt.tight_layout()
plt.show()

In [ ]:
# ============================================================================
# CELL 4: ENHANCED FEATURE EXTRACTION (CORRECTED FOR YOUR 7 CLASSES)
# ============================================================================
import cv2
import numpy as np
from skimage import feature
from scipy import stats
from tqdm import tqdm
import json
import os
import shutil
from pathlib import Path

class EnhancedFeatureExtractor:
    """Extracts multiple feature modalities with YOUR 7 disease classes"""

    def __init__(self):
        self.feature_dir = Path('data/features')
        self.feature_dir.mkdir(exist_ok=True)

    def extract_morphological(self, mask):
        """Extract shape features (10 features)"""
        contours, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)

        if not contours:
            return np.zeros(10, dtype=np.float32)

        cnt = max(contours, key=cv2.contourArea)
        area = cv2.contourArea(cnt)
        perimeter = cv2.arcLength(cnt, True)
        x, y, w, h = cv2.boundingRect(cnt)

        aspect_ratio = w / max(h, 1)
        circularity = (4 * np.pi * area) / max(perimeter ** 2, 1)

        # Hu moments (first 3) for rotation invariance
        hu = cv2.HuMoments(cv2.moments(cnt)).flatten()[:3]
        hu = -np.sign(hu) * np.log10(np.abs(hu) + 1e-10)

        return np.array([
            area, perimeter, aspect_ratio, circularity,
            w, h, (area / (w * h + 1e-10)), *hu
        ], dtype=np.float32)

    def extract_texture_color(self, masked_img, mask):
        """Extract texture and color features (37 features)"""
        nonzero_mask = (mask > 0)
        if not np.any(nonzero_mask):
            return np.zeros(37, dtype=np.float32)

        # 1. Texture (GLCM) - MUST BE 2D
        gray = cv2.cvtColor(masked_img, cv2.COLOR_RGB2GRAY)
        # We pass the full 2D array, not just the pixels
        glcm = feature.graycomatrix(gray, distances=[2], angles=[0],
                                   levels=256, symmetric=True, normed=True)

        glcm_feats = [
            feature.graycoprops(glcm, 'contrast')[0, 0],
            feature.graycoprops(glcm, 'homogeneity')[0, 0],
            feature.graycoprops(glcm, 'energy')[0, 0],
            feature.graycoprops(glcm, 'correlation')[0, 0]
        ]

        # 2. Color Histograms (8 bins per channel = 24 features)
        color_hist = []
        for i in range(3):
            channel_data = masked_img[:, :, i][nonzero_mask]
            hist, _ = np.histogram(channel_data, bins=8, range=(1, 256)) # Skip 0 (bg)
            hist = hist / (hist.sum() + 1e-10)
            color_hist.extend(hist)

        # 3. Color Stats (3 per channel = 9 features)
        color_stats = []
        for i in range(3):
            channel_data = masked_img[:, :, i][nonzero_mask]
            color_stats.extend([
                np.mean(channel_data),
                np.std(channel_data),
                float(stats.skew(channel_data))
            ])

        return np.concatenate([glcm_feats, color_hist, color_stats]).astype(np.float32)

    def extract_red_features(self, masked_img, mask):
        """Disease-specific saturation features (6 features)"""
        nonzero_mask = (mask > 0)
        if not np.any(nonzero_mask):
            return np.zeros(6, dtype=np.float32)

        hsv = cv2.cvtColor(masked_img, cv2.COLOR_RGB2HSV)
        sat = hsv[:, :, 1][nonzero_mask]

        red_ch = masked_img[:, :, 0][nonzero_mask]
        gray_ch = cv2.cvtColor(masked_img, cv2.COLOR_RGB2GRAY)[nonzero_mask]

        red_ratio = np.mean(red_ch) / (np.mean(gray_ch) + 1e-10)

        return np.array([
            np.mean(sat), np.std(sat), np.max(sat),
            np.percentile(sat, 90), np.percentile(sat, 95),
            red_ratio
        ], dtype=np.float32)

    # ============================================================================
    # CRITICAL FIX: YOUR 7 CLASS LABEL EXTRACTION
    # ============================================================================
    def _extract_label_correct(self, img_path):
        """CORRECT label extraction for YOUR 7 tea leaf disease classes"""
        stem = img_path.stem.lower()

        # YOUR 7 SPECIFIC CLASSES - BASED ON YOUR FILENAMES
        # From your debug output: "Helopelties 565", "Gray Blight 1918", "Red Rust 1136", "Algal leaf spot 451"

        # Check in order of priority (most specific first)
        if 'algal' in stem or 'leaf spot' in stem:
            return 0  # Class 0: Algal leaf spot
        elif 'brown' in stem and 'blight' in stem:
            return 1  # Class 1: Brown Blight
        elif 'gray' in stem or 'grey' in stem:
            return 2  # Class 2: Gray Blight
        elif 'healthy' in stem:
            return 3  # Class 3: Healthy
        elif 'helopelties' in stem:
            return 4  # Class 4: Helopelties
        elif 'red rust' in stem:
            return 5  # Class 5: Red Rust
        elif 'red spider' in stem or 'spider' in stem:
            return 6  # Class 6: Red Spider
        elif 'red' in stem:  # Just "red" might be Red Rust
            return 5
        elif 'blight' in stem:  # Just "blight" might be Gray Blight
            return 2
        else:
            # Default based on your debug output distribution
            print(f"⚠️  Could not classify '{stem}', defaulting to Healthy")
            return 3  # Default to Healthy

    def batch_extract(self, save_to_drive=False, use_csv_labels=True):
        """
        Extract features with CORRECT 7-class labels

        Args:
            use_csv_labels: If True, uses the CSV file for accurate labels
        """
        img_paths = [f for f in Path('data/raw_images').glob('*.*')
                    if f.suffix.lower() in ['.jpg', '.png', '.jpeg']]
        mask_dir = Path('data/masks')

        # OPTIONAL: Load labels from CSV for maximum accuracy
        label_dict = None
        if use_csv_labels and os.path.exists('data/tea_leaf_7classes.csv'):
            import pandas as pd
            df = pd.read_csv('data/tea_leaf_7classes.csv')
            label_dict = dict(zip(df['filename'].apply(lambda x: Path(x).stem), df['label']))
            print(f"✅ Loaded {len(label_dict)} labels from CSV")

        all_data = []
        print(f"🔍 Processing {len(img_paths)} images with 7-class labels...")

        for img_path in tqdm(img_paths, desc="Extracting features"):
            mask_path = mask_dir / f"{img_path.stem}_mask.png"
            if not mask_path.exists():
                continue

            img = cv2.imread(str(img_path))
            if img is None:
                continue
            img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
            mask = cv2.imread(str(mask_path), cv2.IMREAD_GRAYSCALE)

            # Ensure mask and image sizes match
            if mask.shape[:2] != img.shape[:2]:
                mask = cv2.resize(mask, (img.shape[1], img.shape[0]))

            masked = cv2.bitwise_and(img_rgb, img_rgb, mask=mask)

            # Extraction
            f_morph = self.extract_morphological(mask)
            f_tex_col = self.extract_texture_color(masked, mask)
            f_red = self.extract_red_features(masked, mask)

            # Combine into one flat vector for ML
            full_vector = np.concatenate([f_morph, f_tex_col, f_red])

            # CORRECT: Get proper label
            if label_dict is not None:
                # Use CSV label if available
                label = label_dict.get(img_path.stem, 3)  # Default to Healthy
            else:
                # Use filename-based extraction
                label = self._extract_label_correct(img_path)

            # Store with separate modalities for fusion models
            all_data.append({
                'image_id': img_path.stem,
                'label': label,  # CORRECT 7-class label (0-6)
                'morphological': f_morph.tolist(),  # 10 features
                'texture_color': f_tex_col.tolist(),  # 37 features
                'red_features': f_red.tolist(),  # 6 features
                'combined_features': full_vector.tolist(),  # 53 features total
                'filename': img_path.name
            })

        # Verify we have 7 classes
        labels = [d['label'] for d in all_data]
        unique_labels = sorted(set(labels))
        print(f"\n✅ Feature extraction complete!")
        print(f"   Total samples: {len(all_data)}")
        print(f"   Unique classes found: {len(unique_labels)}")
        print(f"   Classes: {unique_labels}")

        if len(unique_labels) != 7:
            print(f"⚠️  WARNING: Expected 7 classes but found {len(unique_labels)}")
            # Show distribution
            from collections import Counter
            label_counts = Counter(labels)
            print("   Distribution:")
            for label, count in label_counts.most_common():
                print(f"     Class {label}: {count} samples")

        # Save locally
        out_file = self.feature_dir / 'extracted_features_7class.json'
        with open(out_file, 'w') as f:
            json.dump(all_data, f)

        if save_to_drive:
            drive_path = '/content/drive/MyDrive/tea_leaf_7class_features.json'
            shutil.copy(out_file, drive_path)
            print(f"✅ Features backed up to: {drive_path}")

        return all_data

# Run it with correct labels
print("="*60)
print("RUNNING FEATURE EXTRACTION WITH 7-CLASS LABELS")
print("="*60)

# First, let's create the CSV labels if not exists
if not os.path.exists('data/tea_leaf_7classes.csv'):
    print("Creating 7-class label CSV first...")

    # Simple function to create labels from your filenames
    def create_simple_label_csv():
        import pandas as pd
        from pathlib import Path

        image_dir = Path('data/raw_images')
        image_files = [f for f in image_dir.glob('*.*')
                      if f.suffix.lower() in ['.jpg', '.png', '.jpeg']]

        labels = []
        for img_path in image_files:
            stem = img_path.stem.lower()
            filename = img_path.name

            # Map to your 7 classes
            if 'algal' in stem or 'leaf spot' in stem:
                label = 0
            elif 'brown' in stem and 'blight' in stem:
                label = 1
            elif ('gray' in stem or 'grey' in stem) and 'blight' in stem:
                label = 2
            elif 'healthy' in stem:
                label = 3
            elif 'helopelties' in stem:
                label = 4
            elif 'red rust' in stem:
                label = 5
            elif 'red spider' in stem or 'spider' in stem:
                label = 6
            elif 'red' in stem:
                label = 5  # Assume Red Rust
            elif 'blight' in stem:
                label = 2  # Assume Gray Blight
            else:
                label = 3  # Default to Healthy

            labels.append({
                'filename': filename,
                'label': label,
                'stem': stem
            })

        df = pd.DataFrame(labels)
        df.to_csv('data/tea_leaf_7classes.csv', index=False)

        print(f"✅ Created CSV with {len(df)} labels")
        print(f"   Class distribution:")
        for label in range(7):
            count = len(df[df['label'] == label])
            if count > 0:
                print(f"     Class {label}: {count} samples")

        return df

    label_df = create_simple_label_csv()

# Now extract features with correct labels
extractor = EnhancedFeatureExtractor()
features_data = extractor.batch_extract(save_to_drive=True, use_csv_labels=True)

In [ ]:
# ============================================================================
# CELL 5: INSTALL DEPENDENCIES FOR TRAINING
# ============================================================================
!pip install -q torch torchvision torchaudio --index-url https://download.pytorch.org/whl/cu118
!pip install -q pytorch-lightning albumentations

print("✓ Training dependencies installed")

In [ ]:
# ============================================================================
# CELL 6: FUSION MODEL ARCHITECTURES (CORRECTED)
# ============================================================================
import torch
import torch.nn as nn
import torch.nn.functional as F

class CrossModalAttentionFusion(nn.Module):
    def __init__(self, num_classes=7, embed_dim=128):
        super().__init__()

        # CNN for RGB images - Projects to embed_dim
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1),
            nn.BatchNorm2d(64),
            nn.ReLU(),
            nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1),
            nn.BatchNorm2d(128),
            nn.ReLU(),
            nn.AdaptiveAvgPool2d((1, 1)) # Global Average Pooling
        )
        self.cnn_proj = nn.Linear(128, embed_dim)

        # Encoders for handcrafted features (input sizes match Cell 4)
        self.morph_encoder = nn.Sequential(
            nn.Linear(10, embed_dim), # 10 morphological features
            nn.ReLU(),
            nn.LayerNorm(embed_dim)
        )

        self.texture_encoder = nn.Sequential(
            nn.Linear(37, embed_dim), # 37 texture/color features
            nn.ReLU(),
            nn.LayerNorm(embed_dim)
        )

        self.red_encoder = nn.Sequential(
            nn.Linear(6, embed_dim),  # 6 disease-specific features
            nn.ReLU(),
            nn.LayerNorm(embed_dim)
        )

        # Cross-modal attention
        self.attention = nn.MultiheadAttention(
            embed_dim=embed_dim, num_heads=4, batch_first=True, dropout=0.1
        )

        # Classification head
        self.classifier = nn.Sequential(
            nn.Linear(embed_dim, 256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, num_classes)
        )

    def forward(self, image, morph_feat, texture_feat, red_feat):
        # 1. Extract and project all modalities to [Batch, 1, embed_dim]
        vis = self.cnn_proj(self.cnn(image).view(image.size(0), -1)).unsqueeze(1)
        mor = self.morph_encoder(morph_feat).unsqueeze(1)
        tex = self.texture_encoder(texture_feat).unsqueeze(1)
        red = self.red_encoder(red_feat).unsqueeze(1)

        # 2. Concatenate: [Batch, 4, embed_dim]
        combined = torch.cat([vis, mor, tex, red], dim=1)

        # 3. Attention (Self-attention across modalities)
        attn_out, _ = self.attention(combined, combined, combined)

        # 4. Global average pool across the 4 modality "tokens"
        fused = attn_out.mean(dim=1)

        return self.classifier(fused)

# Test the corrected model
print("Testing corrected architecture...")
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
model = CrossModalAttentionFusion().to(device)

# Mock data matching your extracted features
batch_size = 8
img_mock = torch.randn(batch_size, 3, 224, 224).to(device)
morph_mock = torch.randn(batch_size, 10).to(device)
tex_mock = torch.randn(batch_size, 37).to(device)
red_mock = torch.randn(batch_size, 6).to(device)

output = model(img_mock, morph_mock, tex_mock, red_mock)
print(f"✅ Success! Output shape: {output.shape}") # Expected: [8, 7]

In [ ]:
import albumentations as A
print(f"Albumentations version: {A.__version__}")

In [ ]:
# ============================================================================
# DEBUG CELL: CHECK LABELS IN YOUR DATASET
# ============================================================================

# Add this cell after Cell 4 (feature extraction) to check your labels
print("🔍 DEBUGGING LABEL ISSUE")
print("="*60)

# Load features to check labels
with open('data/features/extracted_features.json', 'r') as f:
    features_data = json.load(f)

# Extract all labels
all_labels = [d['label'] for d in features_data]
unique_labels = sorted(set(all_labels))
label_counts = {label: all_labels.count(label) for label in unique_labels}

print(f"Total samples: {len(features_data)}")
print(f"Unique labels found: {unique_labels}")
print(f"Number of unique labels: {len(unique_labels)}")
print("\nLabel distribution:")
for label in sorted(label_counts.keys()):
    print(f"  Label {label}: {label_counts[label]} samples ({label_counts[label]/len(features_data):.1%})")

# Check what values are in the labels
print("\n" + "="*60)
print("INVESTIGATING LABEL VALUES:")

# Check first few samples
for i, sample in enumerate(features_data[:10]):
    print(f"Sample {i}: image_id='{sample['image_id']}', label={sample['label']}")

# Check filenames to understand pattern
print("\n" + "="*60)
print("CHECKING FILENAME PATTERNS:")

image_dir = Path('data/raw_images')
sample_files = list(image_dir.glob('*.*'))[:10]
for img_file in sample_files:
    print(f"  {img_file.name}")

# The issue is likely in your label extraction function
print("\n" + "="*60)
print("PROBLEM DIAGNOSIS:")
print("Your dataset has only 4 unique labels (0, 1, 2, 3) instead of 7.")
print("This means the label extraction function is not working correctly.")

In [ ]:
# ============================================================================
# CELL 7: DATASET AND DATALOADER (FIXED FOR NEW FEATURE FORMAT)
# ============================================================================
import torch
from torch.utils.data import Dataset, DataLoader
from PIL import Image
import albumentations as A
from albumentations.pytorch import ToTensorV2
import json
from sklearn.model_selection import train_test_split

class TeaLeafDataset(Dataset):
    """Dataset for tea leaf images with 4 modalities"""

    def __init__(self, features_file, image_dir, transform=None, phase='train', img_size=224):
        with open(features_file, 'r') as f:
            self.data = json.load(f)

        self.image_dir = Path(image_dir)
        self.phase = phase
        self.img_size = img_size

        # Default transforms
        if transform is None:
            self.transform = self._get_default_transforms(phase, img_size)
        else:
            self.transform = transform

        # Create label mapping
        all_labels = sorted(set([d['label'] for d in self.data]))
        self.label_to_idx = {label: i for i, label in enumerate(all_labels)}
        self.idx_to_label = {i: label for label, i in self.label_to_idx.items()}
        self.num_classes = len(all_labels)

        # Update labels with indices
        for item in self.data:
            item['label_idx'] = self.label_to_idx[item['label']]

        print(f"Dataset: {len(self.data)} samples, {self.num_classes} classes")

    def _get_default_transforms(self, phase, img_size):
        """Simplified transforms that work with Albumentations 2.0+"""
        if phase == 'train':
            return A.Compose([
                A.Resize(height=img_size, width=img_size),
                A.HorizontalFlip(p=0.5),
                A.Rotate(limit=30, p=0.5),
                A.RandomBrightnessContrast(brightness_limit=0.2, contrast_limit=0.2, p=0.5),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])
        else:
            return A.Compose([
                A.Resize(height=img_size, width=img_size),
                A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
                ToTensorV2()
            ])

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load image
        img_path = self.image_dir / f"{item['image_id']}.jpg"
        if not img_path.exists():
            img_path = self.image_dir / f"{item['image_id']}.png"

        if not img_path.exists():
            # Try to find the image with any extension
            possible_files = list(self.image_dir.glob(f"{item['image_id']}.*"))
            if possible_files:
                img_path = possible_files[0]
            else:
                # Create dummy image if not found
                dummy_img = np.zeros((self.img_size, self.img_size, 3), dtype=np.uint8)
                image = Image.fromarray(dummy_img)
                image = np.array(image)
        else:
            image = Image.open(img_path).convert('RGB')
            image = np.array(image)

        # Apply transforms
        if self.transform:
            try:
                augmented = self.transform(image=image)
                image = augmented['image']
            except Exception as e:
                print(f"Transform error for {img_path}: {e}")
                # Return dummy image if transform fails
                dummy_img = np.zeros((3, self.img_size, self.img_size), dtype=np.float32)
                image = torch.tensor(dummy_img)

        # GET FEATURES FROM SEPARATE MODALITIES (from corrected Cell 4)
        # Your features are stored in separate keys
        if 'morphological' in item and 'texture_color' in item and 'red_features' in item:
            # New format: separate modalities
            morph = torch.tensor(item['morphological'], dtype=torch.float32)
            texture = torch.tensor(item['texture_color'], dtype=torch.float32)
            red = torch.tensor(item['red_features'], dtype=torch.float32)
        elif 'features' in item:
            # Old format: single features array
            full_vector = item['features']
            # Slice based on known sizes
            morph = torch.tensor(full_vector[0:10], dtype=torch.float32)
            texture = torch.tensor(full_vector[10:10+37], dtype=torch.float32)
            red = torch.tensor(full_vector[10+37:10+37+6], dtype=torch.float32)
        else:
            # Create dummy features if none found
            print(f"Warning: No features found for {item['image_id']}")
            morph = torch.zeros(10, dtype=torch.float32)
            texture = torch.zeros(37, dtype=torch.float32)
            red = torch.zeros(6, dtype=torch.float32)

        # Get label
        if 'label_idx' in item:
            label = torch.tensor(item['label_idx'], dtype=torch.long)
        else:
            # Fallback: calculate from label
            label = torch.tensor(self.label_to_idx[item['label']], dtype=torch.long)

        return image, morph, texture, red, label

    def get_class_weights(self):
        """Calculate class weights for imbalance handling"""
        labels = []
        for d in self.data:
            if 'label_idx' in d:
                labels.append(d['label_idx'])
            else:
                # Fallback: calculate from label
                labels.append(self.label_to_idx[d['label']])

        class_counts = np.bincount(labels)
        class_counts = np.maximum(class_counts, 1)  # Avoid division by zero

        # Inverse frequency weighting
        weights = 1.0 / class_counts
        weights = weights / weights.sum() * len(class_counts)

        return torch.tensor(weights, dtype=torch.float32)

def create_dataloaders(features_file='data/features/extracted_features_7class.json',
                      image_dir='data/raw_images',
                      batch_size=32,
                      test_size=0.2,
                      val_size=0.15,
                      img_size=224):
    """Create train/val/test dataloaders"""

    print(f"Loading features from: {features_file}")

    # Load data
    with open(features_file, 'r') as f:
        all_data = json.load(f)

    # Check feature format
    sample_item = all_data[0]
    print(f"Sample item keys: {list(sample_item.keys())}")

    if 'morphological' in sample_item:
        print("✓ Using new format: separate modalities")
        print(f"  Morphological features: {len(sample_item['morphological'])}")
        print(f"  Texture/Color features: {len(sample_item['texture_color'])}")
        print(f"  Red features: {len(sample_item['red_features'])}")
    elif 'features' in sample_item:
        print("⚠️ Using old format: combined features array")
    else:
        print("❌ Unknown feature format!")
        return None

    # Extract labels for stratification
    labels = [d['label'] for d in all_data]
    unique_labels = sorted(set(labels))
    print(f"\nUnique labels found: {unique_labels}")
    print(f"Number of classes: {len(unique_labels)}")

    # First split: train (80%) + temp (20%)
    indices = list(range(len(all_data)))
    train_idx, temp_idx = train_test_split(
        indices, test_size=test_size, stratify=labels, random_state=42
    )

    # Second split: val (15%) + test (5%)
    temp_labels = [labels[i] for i in temp_idx]
    val_idx, test_idx = train_test_split(
        temp_idx, test_size=0.25, stratify=temp_labels, random_state=42
    )

    # Create data subsets
    train_data = [all_data[i] for i in train_idx]
    val_data = [all_data[i] for i in val_idx]
    test_data = [all_data[i] for i in test_idx]

    print(f"\nData split:")
    print(f"  Train: {len(train_data)} samples ({len(train_data)/len(all_data):.1%})")
    print(f"  Val: {len(val_data)} samples ({len(val_data)/len(all_data):.1%})")
    print(f"  Test: {len(test_data)} samples ({len(test_data)/len(all_data):.1%})")

    # Create datasets WITH label_idx calculation
    def add_label_indices(data):
        """Add label_idx to data"""
        # First get all labels
        all_labels = sorted(set([d['label'] for d in data]))
        label_to_idx = {label: i for i, label in enumerate(all_labels)}

        # Add label_idx to each item
        for item in data:
            item['label_idx'] = label_to_idx[item['label']]
        return data

    # Add label indices
    train_data = add_label_indices(train_data)
    val_data = add_label_indices(val_data)
    test_data = add_label_indices(test_data)

    # Create datasets
    train_dataset = TeaLeafDataset(
        features_file, image_dir, phase='train', img_size=img_size
    )
    train_dataset.data = train_data
    # Update label mapping for train dataset
    train_dataset.label_to_idx = {label: i for i, label in enumerate(sorted(set([d['label'] for d in train_data])))}
    train_dataset.idx_to_label = {i: label for label, i in train_dataset.label_to_idx.items()}
    train_dataset.num_classes = len(train_dataset.label_to_idx)

    val_dataset = TeaLeafDataset(
        features_file, image_dir, phase='val', img_size=img_size
    )
    val_dataset.data = val_data
    val_dataset.label_to_idx = train_dataset.label_to_idx  # Use same mapping as train
    val_dataset.idx_to_label = train_dataset.idx_to_label
    val_dataset.num_classes = train_dataset.num_classes

    test_dataset = TeaLeafDataset(
        features_file, image_dir, phase='test', img_size=img_size
    )
    test_dataset.data = test_data
    test_dataset.label_to_idx = train_dataset.label_to_idx  # Use same mapping as train
    test_dataset.idx_to_label = train_dataset.idx_to_label
    test_dataset.num_classes = train_dataset.num_classes

    # Get class weights (from train dataset only)
    class_weights = train_dataset.get_class_weights()
    print(f"\nClass weights: {class_weights.numpy().round(3)}")

    # Show class distribution
    print("\nClass distribution in training set:")
    train_labels = [d['label'] for d in train_data]
    from collections import Counter
    label_counts = Counter(train_labels)
    for label, count in sorted(label_counts.items()):
        print(f"  Class {label}: {count} samples ({count/len(train_data):.1%})")

    # Create dataloaders
    train_loader = DataLoader(
        train_dataset, batch_size=batch_size, shuffle=True,
        num_workers=2, pin_memory=True, drop_last=True
    )

    val_loader = DataLoader(
        val_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )

    test_loader = DataLoader(
        test_dataset, batch_size=batch_size, shuffle=False,
        num_workers=2, pin_memory=True
    )

    return train_loader, val_loader, test_loader, class_weights

# Test the dataloader
print("Creating dataloaders...")
print("="*60)

# Try loading from different possible feature files
feature_files_to_try = [
    'data/features/extracted_features_7class.json',  # New format
    '/content/drive/MyDrive/tea_leaf_7class_features.json',  # Google Drive backup
    'data/features/extracted_features.json'  # Old format
]

for feature_file in feature_files_to_try:
    if os.path.exists(feature_file):
        print(f"Trying feature file: {feature_file}")
        try:
            train_loader, val_loader, test_loader, class_weights = create_dataloaders(
                features_file=feature_file,
                batch_size=8  # Small for testing
            )
            print(f"✓ Successfully loaded from {feature_file}")
            break
        except Exception as e:
            print(f"✗ Failed to load from {feature_file}: {e}")
            continue
else:
    print("❌ No feature file found!")
    # Create dummy dataloaders for testing
    train_loader, val_loader, test_loader, class_weights = None, None, None, None

if train_loader is not None:
    # Show a batch
    print("\n" + "="*60)
    print("Sample batch:")
    batch = next(iter(train_loader))
    images, morph, texture, red, labels = batch
    print(f"Images: {images.shape}")
    print(f"Morph features: {morph.shape}")
    print(f"Texture features: {texture.shape}")
    print(f"Red features: {red.shape}")
    print(f"Labels: {labels.shape}")
    print(f"Label values: {labels[:4].numpy()}")

    # Visualize
    fig, axes = plt.subplots(2, 4, figsize=(15, 8))
    for i in range(min(8, len(images))):
        ax = axes[i//4, i%4]
        img = images[i].permute(1, 2, 0).numpy()
        img = img * np.array([0.229, 0.224, 0.225]) + np.array([0.485, 0.456, 0.406])
        img = np.clip(img, 0, 1)
        ax.imshow(img)
        ax.set_title(f"Class: {labels[i].item()}")
        ax.axis('off')
    plt.tight_layout()
    plt.show()
else:
    print("❌ Could not create dataloaders. Please check feature files.")

In [ ]:
# ============================================================================
# CELL 8: TRAINING SYSTEM (FIXED - DISABLE PROGRESS BAR IN NOTEBOOKS)
# ============================================================================
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, LearningRateMonitor
import torch
import torch.nn as nn
import torch.optim as optim
from torch.optim.lr_scheduler import CosineAnnealingLR
import torchmetrics
import os
import shutil
from pathlib import Path
import json
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from sklearn.metrics import classification_report, confusion_matrix

class TeaLeafFusionSystem(pl.LightningModule):
    """Robust training system compatible with multiple fusion models"""

    def __init__(self, model, num_classes=7, learning_rate=1e-4,
                 class_weights=None, model_name='fusion'):
        super().__init__()
        self.model = model
        self.num_classes = num_classes
        self.lr = learning_rate
        self.model_name = model_name

        # Loss function
        if class_weights is not None:
            class_weights = torch.tensor(class_weights, dtype=torch.float)
            self.criterion = nn.CrossEntropyLoss(weight=class_weights.to(self.device),
                                                label_smoothing=0.1)
        else:
            self.criterion = nn.CrossEntropyLoss(label_smoothing=0.1)

        # Metrics
        self.train_acc = torchmetrics.Accuracy(task='multiclass', num_classes=num_classes)
        self.val_acc   = torchmetrics.Accuracy(task='multiclass', num_classes=num_classes)
        self.test_acc  = torchmetrics.Accuracy(task='multiclass', num_classes=num_classes)

        self.train_f1 = torchmetrics.F1Score(task='multiclass', num_classes=num_classes, average='macro')
        self.val_f1   = torchmetrics.F1Score(task='multiclass', num_classes=num_classes, average='macro')
        self.test_f1  = torchmetrics.F1Score(task='multiclass', num_classes=num_classes, average='macro')

        # Test analysis storage
        self.test_predictions = []
        self.test_labels = []
        self.test_gates = []

    def forward(self, image, morph, texture, red):
        return self.model(image, morph, texture, red)

    def _process_outputs(self, outputs):
        if isinstance(outputs, tuple) and len(outputs) == 3:
            logits, _, gates = outputs
            return logits, gates
        else:
            return outputs, None

    def training_step(self, batch, batch_idx):
        image, morph, texture, red, labels = batch
        outputs = self(image, morph, texture, red)

        logits, gates = self._process_outputs(outputs)

        loss = self.criterion(logits, labels)

        self.log('train_loss', loss, prog_bar=True, on_step=True, on_epoch=True)
        self.train_acc(logits, labels)
        self.train_f1(logits, labels)
        self.log('train_acc', self.train_acc, prog_bar=True)
        self.log('train_f1', self.train_f1, prog_bar=False)

        if gates is not None:
            for i in range(gates.shape[1]):
                self.log(f'gate_{i}', gates[:, i].mean(), prog_bar=False)

        return loss

    def validation_step(self, batch, batch_idx):
        image, morph, texture, red, labels = batch
        outputs = self(image, morph, texture, red)

        logits, _ = self._process_outputs(outputs)

        loss = self.criterion(logits, labels)

        self.log('val_loss', loss, prog_bar=True)
        self.val_acc(logits, labels)
        self.val_f1(logits, labels)
        self.log('val_acc', self.val_acc, prog_bar=True)
        self.log('val_f1', self.val_f1, prog_bar=False)

        return loss

    def test_step(self, batch, batch_idx):
        image, morph, texture, red, labels = batch

        if hasattr(self.model, 'dropout'):
            self.model.dropout.train()

        preds_list = []
        gates_list = []

        for _ in range(5):
            outputs = self(image, morph, texture, red)
            logits, gates = self._process_outputs(outputs)

            preds_list.append(torch.softmax(logits, dim=1))
            if gates is not None:
                gates_list.append(gates)

        avg_preds = torch.stack(preds_list).mean(0)

        self.test_predictions.append(avg_preds.cpu())
        self.test_labels.append(labels.cpu())
        if gates_list:
            avg_gates = torch.stack(gates_list).mean(0)
            self.test_gates.append(avg_gates.cpu())

        self.test_acc(avg_preds, labels)
        self.test_f1(avg_preds, labels)

        return avg_preds

    def on_test_epoch_end(self):
        if not self.test_predictions:
            return

        all_preds = torch.cat(self.test_predictions)
        all_labels = torch.cat(self.test_labels)
        pred_classes = torch.argmax(all_preds, dim=1)

        report = classification_report(
            all_labels.numpy(),
            pred_classes.numpy(),
            output_dict=True,
            zero_division=0
        )

        cm = confusion_matrix(all_labels.numpy(), pred_classes.numpy())

        results = {
            'model': self.model_name,
            'accuracy': report['accuracy'],
            'macro_f1': report['macro avg']['f1-score'],
            'weighted_f1': report['weighted avg']['f1-score'],
            'per_class': {str(i): report[str(i)] for i in range(self.num_classes)},
            'confusion_matrix': cm.tolist()
        }

        os.makedirs('results', exist_ok=True)
        results_file = f'results/{self.model_name}_results.json'
        with open(results_file, 'w') as f:
            json.dump(results, f, indent=2)

        drive_dir = '/content/drive/MyDrive/tea_leaf_results'
        if Path(drive_dir).exists():
            os.makedirs(drive_dir, exist_ok=True)
            shutil.copy(results_file, f'{drive_dir}/{self.model_name}_results.json')

        print(f"\n{'='*60}")
        print(f"FINAL TEST RESULTS - {self.model_name.upper()}")
        print(f"{'='*60}")
        print(f"Accuracy:     {report['accuracy']:.4f}")
        print(f"Macro F1:     {report['macro avg']['f1-score']:.4f}")
        print(f"Weighted F1:  {report['weighted avg']['f1-score']:.4f}")

        plt.figure(figsize=(8, 6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues', cbar=False)
        plt.title(f'Confusion Matrix - {self.model_name}')
        plt.ylabel('True Label')
        plt.xlabel('Predicted Label')
        plt.tight_layout()
        plt.savefig(f'results/{self.model_name}_confusion.png', dpi=150)
        plt.show()

        if self.test_gates:
            all_gates = torch.cat(self.test_gates).numpy()
            self._plot_gate_importance(all_gates, all_labels.numpy())

    def _plot_gate_importance(self, gate_weights, labels):
        unique_labels = np.unique(labels)
        modality_names = ['Visual', 'Morphological', 'Texture', 'Red Features']

        fig, axes = plt.subplots(2, 2, figsize=(12, 10))
        axes = axes.flatten()

        for i, (ax, name) in enumerate(zip(axes, modality_names)):
            means = []
            stds = []
            for lbl in unique_labels:
                mask = (labels == lbl)
                if mask.any():
                    means.append(gate_weights[mask, i].mean())
                    stds.append(gate_weights[mask, i].std())
                else:
                    means.append(0)
                    stds.append(0)

            ax.bar(range(len(unique_labels)), means, yerr=stds, capsize=5, color=f'C{i}', alpha=0.8)
            ax.set_title(f'{name} Modality Importance')
            ax.set_xlabel('Class')
            ax.set_ylabel('Average Gate Weight')
            ax.set_xticks(range(len(unique_labels)))
            ax.set_xticklabels([f'C{int(l)}' for l in unique_labels])

        plt.tight_layout()
        plt.savefig(f'results/{self.model_name}_gate_importance.png', dpi=150)
        plt.show()

    def configure_optimizers(self):
        optimizer = optim.AdamW(
            self.parameters(),
            lr=self.lr,
            weight_decay=1e-4,
            betas=(0.9, 0.999)
        )

        scheduler = CosineAnnealingLR(
            optimizer,
            T_max=self.trainer.max_epochs,
            eta_min=1e-6
        )

        return {
            'optimizer': optimizer,
            'lr_scheduler': {
                'scheduler': scheduler,
                'interval': 'epoch',
                'frequency': 1
            }
        }

def train_model(model_type='cross_attention', num_epochs=50, batch_size=32, save_to_drive=True):
    print(f"\n{'='*60}")
    print(f"TRAINING {model_type.upper()} MODEL | {num_epochs} EPOCHS")
    print(f"{'='*60}")

    train_loader, val_loader, test_loader, class_weights = create_dataloaders(batch_size=batch_size)

    if model_type == 'cross_attention':
        model = CrossModalAttentionFusion(num_classes=7)
        model_name = 'CrossModalAttention'
    elif model_type == 'experts_gating':
        model = ModalityExpertsWithGating(num_classes=7)
        model_name = 'ExpertsWithGating'
    else:
        raise ValueError(f"Unknown model_type: {model_type}")

    system = TeaLeafFusionSystem(
        model=model,
        num_classes=7,
        learning_rate=1e-4,
        class_weights=class_weights,
        model_name=model_name
    )

    checkpoint_callback = ModelCheckpoint(
        dirpath='checkpoints',
        filename=f'{model_name}-{{epoch:02d}}-{{val_loss:.3f}}',
        monitor='val_loss',
        mode='min',
        save_top_k=2,
        save_last=True
    )

    early_stop = EarlyStopping(
        monitor='val_loss',
        patience=15,
        mode='min',
        verbose=True
    )

    lr_monitor = LearningRateMonitor(logging_interval='epoch')

    # Critical fix: Disable progress bar in Jupyter/Colab notebooks to avoid Rich recursion bug
    trainer = pl.Trainer(
        max_epochs=num_epochs,
        accelerator='auto',
        devices=1,
        callbacks=[checkpoint_callback, early_stop, lr_monitor],
        log_every_n_steps=10,
        check_val_every_n_epoch=1,
        enable_progress_bar=False,  # ← This prevents the RecursionError in notebooks
        deterministic=False
    )

    trainer.fit(system, train_loader, val_loader)

    if save_to_drive:
        drive_dir = '/content/drive/MyDrive/tea_leaf_checkpoints'
        if Path(drive_dir).exists():
            os.makedirs(drive_dir, exist_ok=True)
            for ckpt in Path('checkpoints').glob('*.ckpt'):
                shutil.copy(ckpt, drive_dir / ckpt.name)
            print(f"Checkpoints saved to Google Drive: {drive_dir}")

    trainer.test(system, test_loader, ckpt_path='best')

    return system, trainer

# Quick test
if __name__ == "__main__":
    print("Running quick 2-epoch test...")
    test_system, _ = train_model(
        model_type='cross_attention',
        num_epochs=2,
        batch_size=8,
        save_to_drive=False
    )

In [ ]:
# ============================================================================
# FINAL MASTER CELL: DYNAMIC DIMENSION DETECTION & TRAINING
# ============================================================================
import os
os.environ["PL_DISABLE_RICH"] = "1"  # Fix Colab crash

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader
from torch.optim.lr_scheduler import CosineAnnealingLR
import pytorch_lightning as pl
from pytorch_lightning.callbacks import ModelCheckpoint, EarlyStopping, TQDMProgressBar
import torchmetrics
import albumentations as A
from albumentations.pytorch import ToTensorV2
from sklearn.model_selection import train_test_split
from sklearn.metrics import classification_report, confusion_matrix
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import json
import shutil
from pathlib import Path
from PIL import Image

# ----------------------------------------------------------------------------
# 1. FIXED TRANSFORMS
# ----------------------------------------------------------------------------
def get_transforms(phase, img_size=224):
    if phase == 'train':
        return A.Compose([
            A.RandomResizedCrop(size=(img_size, img_size), scale=(0.8, 1.0)),
            A.HorizontalFlip(p=0.5),
            A.Rotate(limit=30, p=0.5),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, p=0.5),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])
    else:
        return A.Compose([
            A.Resize(height=img_size, width=img_size),
            A.Normalize(mean=[0.485, 0.456, 0.406], std=[0.229, 0.224, 0.225]),
            ToTensorV2()
        ])

# ----------------------------------------------------------------------------
# 2. DATASET
# ----------------------------------------------------------------------------
class TeaLeafDataset(Dataset):
    def __init__(self, data_list, image_dir, transform=None):
        self.data = data_list
        self.image_dir = Path(image_dir)
        self.transform = transform

    def __len__(self):
        return len(self.data)

    def __getitem__(self, idx):
        item = self.data[idx]

        # Load Image
        img_path = self.image_dir / f"{item['image_id']}.jpg"
        if not img_path.exists():
            img_path = self.image_dir / f"{item['image_id']}.png"

        try:
            image = np.array(Image.open(img_path).convert('RGB'))
        except:
            image = np.zeros((224, 224, 3), dtype=np.uint8)

        if self.transform:
            image = self.transform(image=image)['image']

        # Load Features
        morph = torch.tensor(item['morphological'], dtype=torch.float32)
        tex = torch.tensor(item['texture_color'], dtype=torch.float32)
        red = torch.tensor(item['red_features'], dtype=torch.float32)

        # Handle Label
        if 'label_idx' in item:
            label = torch.tensor(item['label_idx'], dtype=torch.long)
        else:
            label = torch.tensor(item['label'], dtype=torch.long)

        return image, morph, tex, red, label

def create_dataloaders_fixed(features_file, image_dir, batch_size=32):
    with open(features_file, 'r') as f:
        all_data = json.load(f)

    # Ensure labels are mapped to indices
    labels = [d['label'] for d in all_data]
    unique_labels = sorted(list(set(labels)))
    label_map = {l: i for i, l in enumerate(unique_labels)}

    for d in all_data:
        d['label_idx'] = label_map[d['label']]

    # Detect Dimensions dynamically
    sample = all_data[0]
    dims = {
        'morph': len(sample['morphological']),
        'tex': len(sample['texture_color']),
        'red': len(sample['red_features'])
    }
    print(f"✓ Detected Feature Dims: Morph={dims['morph']}, Tex={dims['tex']}, Red={dims['red']}")

    # Splits
    train_data, temp_data = train_test_split(all_data, test_size=0.2, stratify=[d['label_idx'] for d in all_data], random_state=42)
    val_data, test_data = train_test_split(temp_data, test_size=0.25, stratify=[d['label_idx'] for d in temp_data], random_state=42)

    # Weights
    label_list = [d['label_idx'] for d in train_data]
    counts = np.bincount(label_list)
    weights = 1.0 / np.maximum(counts, 1)
    weights = torch.tensor(weights / weights.sum() * len(counts), dtype=torch.float32)

    # Loaders
    train_dl = DataLoader(TeaLeafDataset(train_data, image_dir, get_transforms('train')), batch_size=batch_size, shuffle=True, num_workers=2, pin_memory=True)
    val_dl = DataLoader(TeaLeafDataset(val_data, image_dir, get_transforms('val')), batch_size=batch_size, shuffle=False, num_workers=2)
    test_dl = DataLoader(TeaLeafDataset(test_data, image_dir, get_transforms('val')), batch_size=batch_size, shuffle=False, num_workers=2)

    return train_dl, val_dl, test_dl, weights, dims

# ----------------------------------------------------------------------------
# 3. MODELS (DYNAMIC INPUT SIZES)
# ----------------------------------------------------------------------------
class CrossModalAttentionFusion(nn.Module):
    def __init__(self, num_classes=7, morph_dim=10, tex_dim=37, red_dim=6):
        super().__init__()
        self.cnn = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.BatchNorm2d(64), nn.ReLU(), nn.MaxPool2d(2),
            nn.Conv2d(64, 128, 3, padding=1), nn.BatchNorm2d(128), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1))
        )
        self.cnn_proj = nn.Linear(128, 128)

        # Encoders use dynamic input dimensions
        self.morph_enc = nn.Sequential(nn.Linear(morph_dim, 64), nn.ReLU(), nn.Linear(64, 128), nn.BatchNorm1d(128))
        self.tex_enc = nn.Sequential(nn.Linear(tex_dim, 128), nn.BatchNorm1d(128))
        self.red_enc = nn.Sequential(nn.Linear(red_dim, 32), nn.ReLU(), nn.Linear(32, 128), nn.BatchNorm1d(128))

        self.attn = nn.MultiheadAttention(embed_dim=128, num_heads=4, batch_first=True, dropout=0.1)
        self.clf = nn.Sequential(nn.Linear(128, 64), nn.ReLU(), nn.Dropout(0.5), nn.Linear(64, num_classes))

    def forward(self, img, morph, tex, red):
        cnn = self.cnn_proj(self.cnn(img).view(img.size(0), -1)).unsqueeze(1)
        m = self.morph_enc(morph).unsqueeze(1)
        t = self.tex_enc(tex).unsqueeze(1)
        r = self.red_enc(red).unsqueeze(1)

        seq = torch.cat([cnn, m, t, r], dim=1)
        attn_out, _ = self.attn(seq, seq, seq)
        return self.clf(attn_out.mean(dim=1)), None

class ModalityExpertsWithGating(nn.Module):
    def __init__(self, num_classes=7, morph_dim=10, tex_dim=37, red_dim=6):
        super().__init__()
        self.vis_exp = nn.Sequential(
            nn.Conv2d(3, 64, 3, padding=1), nn.ReLU(), nn.MaxPool2d(2),
            nn.AdaptiveAvgPool2d((1, 1)), nn.Flatten(), nn.Linear(64, num_classes)
        )
        self.morph_exp = nn.Sequential(nn.Linear(morph_dim, 32), nn.ReLU(), nn.Linear(32, num_classes))
        self.tex_exp = nn.Sequential(nn.Linear(tex_dim, 64), nn.ReLU(), nn.Linear(64, num_classes))
        self.red_exp = nn.Sequential(nn.Linear(red_dim, 16), nn.ReLU(), nn.Linear(16, num_classes))

        # Gating input size varies based on features
        total_dim = morph_dim + tex_dim + red_dim
        self.gate = nn.Sequential(nn.Linear(total_dim, 64), nn.ReLU(), nn.Linear(64, 4), nn.Softmax(dim=1))

    def forward(self, img, morph, tex, red):
        o_v = self.vis_exp(img)
        o_m = self.morph_exp(morph)
        o_t = self.tex_exp(tex)
        o_r = self.red_exp(red)
        g = self.gate(torch.cat([morph, tex, red], dim=1))
        return (g[:,0:1]*o_v + g[:,1:2]*o_m + g[:,2:3]*o_t + g[:,3:4]*o_r), g

# ----------------------------------------------------------------------------
# 4. LIGHTNING SYSTEM
# ----------------------------------------------------------------------------
class TeaLeafSystem(pl.LightningModule):
    def __init__(self, model, num_classes=7, weights=None, name='fusion'):
        super().__init__()
        self.model = model
        self.name = name
        self.num_classes = num_classes
        self.criterion = nn.CrossEntropyLoss(weight=weights)
        self.acc = torchmetrics.Accuracy(task='multiclass', num_classes=num_classes)
        self.f1 = torchmetrics.F1Score(task='multiclass', num_classes=num_classes, average='macro')
        self.test_preds = []
        self.test_labels = []

    def forward(self, img, morph, tex, red):
        return self.model(img, morph, tex, red)

    def training_step(self, batch, batch_idx):
        img, morph, tex, red, y = batch
        out = self(img, morph, tex, red)
        logits = out[0] if isinstance(out, tuple) else out
        loss = self.criterion(logits, y)
        self.log('train_loss', loss, prog_bar=True)
        self.log('train_acc', self.acc(logits, y), prog_bar=True)
        return loss

    def validation_step(self, batch, batch_idx):
        img, morph, tex, red, y = batch
        out = self(img, morph, tex, red)
        logits = out[0] if isinstance(out, tuple) else out
        loss = self.criterion(logits, y)
        self.log('val_loss', loss, prog_bar=True)
        self.log('val_acc', self.acc(logits, y), prog_bar=True)
        return loss

    def test_step(self, batch, batch_idx):
        img, morph, tex, red, y = batch
        out = self(img, morph, tex, red)
        logits = out[0] if isinstance(out, tuple) else out
        preds = torch.argmax(logits, dim=1)
        self.test_preds.append(preds.cpu())
        self.test_labels.append(y.cpu())
        self.acc(logits, y)
        self.f1(logits, y)
        self.log('test_acc', self.acc)

    def on_test_epoch_end(self):
        all_preds = torch.cat(self.test_preds).numpy()
        all_labels = torch.cat(self.test_labels).numpy()
        report = classification_report(all_labels, all_preds, output_dict=True, zero_division=0)
        print(f"\nModel: {self.name}")
        print(f"Test Accuracy: {report['accuracy']:.4f}")
        print(f"Test F1 (Macro): {report['macro avg']['f1-score']:.4f}")

        cm = confusion_matrix(all_labels, all_preds)
        plt.figure(figsize=(8,6))
        sns.heatmap(cm, annot=True, fmt='d', cmap='Blues')
        plt.title(f'Confusion Matrix: {self.name}')
        plt.show()

    def configure_optimizers(self):
        optimizer = optim.AdamW(self.parameters(), lr=1e-3, weight_decay=1e-4)
        scheduler = CosineAnnealingLR(optimizer, T_max=self.trainer.max_epochs)
        return [optimizer], [scheduler]

# ----------------------------------------------------------------------------
# 5. EXECUTION
# ----------------------------------------------------------------------------
def train_experiment(model_type='cross_attention', epochs=10, batch_size=32):
    print(f"🚀 Starting {model_type} experiment...")

    # Path
    feat_file = 'data/features/extracted_features_7class.json'
    if not os.path.exists(feat_file):
        print(f"⚠ Warning: {feat_file} not found. Trying fallback...")
        feat_file = 'data/features/extracted_features.json'

    print(f"📂 Loading features from: {feat_file}")

    # Get loaders AND dimensions
    train_dl, val_dl, test_dl, weights, dims = create_dataloaders_fixed(feat_file, 'data/raw_images', batch_size)

    # Init Model using DETECTED dimensions
    if model_type == 'cross_attention':
        model = CrossModalAttentionFusion(
            num_classes=7,
            morph_dim=dims['morph'],
            tex_dim=dims['tex'],
            red_dim=dims['red']
        )
    else:
        model = ModalityExpertsWithGating(
            num_classes=7,
            morph_dim=dims['morph'],
            tex_dim=dims['tex'],
            red_dim=dims['red']
        )

    system = TeaLeafSystem(model, num_classes=7, weights=weights, name=model_type)

    ckpt = ModelCheckpoint(dirpath='checkpoints', monitor='val_loss', mode='min', save_top_k=1)
    pbar = TQDMProgressBar(refresh_rate=20)
    early_stop = EarlyStopping(monitor='val_loss', patience=5, mode='min')

    trainer = pl.Trainer(
        max_epochs=epochs,
        accelerator='auto',
        callbacks=[ckpt, pbar, early_stop],
        enable_progress_bar=True
    )

    trainer.fit(system, train_dl, val_dl)

    # Save to Drive
    drive_dir = Path('/content/drive/MyDrive/tea_leaf_checkpoints')
    if drive_dir.exists() and ckpt.best_model_path:
        shutil.copy(ckpt.best_model_path, drive_dir / Path(ckpt.best_model_path).name)
        print(f"✓ Saved best model to Google Drive")

    print("\n📊 Running Final Evaluation...")
    trainer.test(system, test_dl)
    return system

if __name__ == "__main__":
    train_experiment(model_type='cross_attention', epochs=15, batch_size=32)